# WaveNet — A Generative Model for Raw Audio (2016)

_Papers · Speech & Audio_

**Generate a raw audio waveform one sample at a time, where each sample is a 256-way classifier that may only look at the samples already produced — and reach far back in time cheaply using dilated causal convolutions whose receptive field grows exponentially with depth.**

---

This notebook is a practice scaffold for the **WaveNet — A Generative Model for Raw Audio (2016)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# --- 0. Sanity-check the worked example: one causal conv step (k=2, d=1). --------------
#     y_t = w0 * x_{t-1} + w1 * x_t,  left-pad by (k-1)*d = 1 zero so it's causal.
x  = torch.tensor([3., 1., 4., 2.]).view(1, 1, -1)     # (batch, channel, time)
w  = torch.tensor([0.5, 2.0]).view(1, 1, 2)            # taps: [past, current]
xp = F.pad(x, (1, 0))                                  # LEFT pad one zero, none on right
y  = F.conv1d(xp, w).view(-1)
print("causal conv output:", y.tolist())               # [6.0, 3.5, 8.5, 6.0]
assert torch.allclose(y, torch.tensor([6.0, 3.5, 8.5, 6.0]))

# --- 1. CausalConv1d: left-pad (k-1)*dilation, convolve with padding=0. ----------------
class CausalConv1d(nn.Module):
    def __init__(self, in_c, out_c, k=2, dilation=1):
        super().__init__()
        self.pad = (k - 1) * dilation                  # how far this layer reaches back
        self.conv = nn.Conv1d(in_c, out_c, k, dilation=dilation, padding=0)
    def forward(self, x):
        return self.conv(F.pad(x, (self.pad, 0)))      # left only -> output t sees inputs <= t

# --- 2. Measure the receptive field empirically with a gradient probe. -----------------
#     Feed a length-L input; backprop d(last output)/d(input); count nonzero-grad inputs.
def receptive_field(dilations, k=2, L=64):
    layers = [CausalConv1d(1, 1, k, d) for d in dilations]
    net = nn.Sequential(*layers)
    for p in net.parameters():                         # weights=1, bias=0 => grad flows
        nn.init.constant_(p, 1.0 if p.dim() > 1 else 0.0)
    inp = torch.zeros(1, 1, L, requires_grad=True)
    out = net(inp)
    out[0, 0, -1].backward()                           # gradient of the LAST output sample
    return int((inp.grad[0, 0].abs() > 0).sum().item())

rf_dilated = receptive_field([1, 2, 4, 8])             # exponential: 2^4
rf_plain   = receptive_field([1, 1, 1, 1])             # linear:      1+4
print(f"receptive field  dilated(1,2,4,8) = {rf_dilated}   plain(1,1,1,1) = {rf_plain}")
assert rf_dilated == 16 and rf_plain == 5              # matches R = 1 + sum (k-1)d

# --- 3. Tiny gated WaveNet: gated residual blocks + skip connections + softmax head. ---
Q = 16                                                 # toy quantization levels (paper: 256)
class GatedBlock(nn.Module):
    def __init__(self, ch, dilation):
        super().__init__()
        self.filt = CausalConv1d(ch, ch, 2, dilation)  # tanh "content" branch
        self.gate = CausalConv1d(ch, ch, 2, dilation)  # sigmoid "gate" branch
        self.res  = nn.Conv1d(ch, ch, 1)               # 1x1 residual projection
        self.skip = nn.Conv1d(ch, ch, 1)               # 1x1 skip projection
    def forward(self, x):
        z = torch.tanh(self.filt(x)) * torch.sigmoid(self.gate(x))   # Eq. 2
        return x + self.res(z), self.skip(z)           # residual out, skip out

class TinyWaveNet(nn.Module):
    def __init__(self, ch=32, dilations=(1, 2, 4, 8)):
        super().__init__()
        self.embed  = nn.Conv1d(1, ch, 1)
        self.blocks = nn.ModuleList(GatedBlock(ch, d) for d in dilations)
        self.head   = nn.Sequential(nn.ReLU(), nn.Conv1d(ch, ch, 1),
                                    nn.ReLU(), nn.Conv1d(ch, Q, 1))   # -> Q-way logits/step
    def forward(self, x):                              # x: (B,1,T) float in [-1,1]
        h, skips = self.embed(x), 0
        for b in self.blocks:
            h, s = b(h); skips = skips + s
        return self.head(skips)                        # (B, Q, T) logits

# --- 4. Toy waveform: a quantized sine wave; train per-sample cross-entropy. ------------
def mu_law(x, mu=Q - 1):                               # mu-law companding (Sec 2.2), mu=Q-1
    return torch.sign(x) * torch.log1p(mu * x.abs()) / torch.log1p(torch.tensor(float(mu)))
T = 256
t = torch.linspace(0, 8 * 3.14159, T)
wave = 0.9 * torch.sin(t)                              # raw signal in [-1,1]
comp = mu_law(wave)                                    # companded
q = ((comp * 0.5 + 0.5) * (Q - 1)).round().long().clamp(0, Q - 1)   # -> {0..Q-1} classes
xf = (q.float() / (Q - 1) * 2 - 1).view(1, 1, T)       # model input (float)

net = TinyWaveNet()
opt = torch.optim.Adam(net.parameters(), lr=3e-3)
for step in range(300):
    logits = net(xf)[..., :-1]                         # predict sample t+1 from <= t
    loss = F.cross_entropy(logits.reshape(Q, -1).T, q[1:].view(-1))
    opt.zero_grad(); loss.backward(); opt.step()
print(f"final per-sample cross-entropy: {loss.item():.4f}  (Q={Q} levels)")

# --- 5. Generate a tiny waveform autoregressively, sample-by-sample. -------------------
@torch.no_grad()
def generate(net, seed, n=64):
    buf = seed.clone()                                 # (1,1,t0) float
    for _ in range(n):
        logits = net(buf)[0, :, -1]                    # Q-way dist for the NEXT sample
        cls = torch.distributions.Categorical(logits=logits).sample()
        nxt = (cls.float() / (Q - 1) * 2 - 1).view(1, 1, 1)
        buf = torch.cat([buf, nxt], dim=-1)            # append, then continue
    return buf[0, 0]
gen = generate(net, xf[..., :16], n=64)
print("generated waveform (first 12 samples):", [round(v, 2) for v in gen[:12].tolist()])
# Training is one parallel pass; generation is strictly sequential (append each sample).
# (Our small CPU run on a toy sine, not the paper's reported MOS / audio.)

## Visualize the data & results

_Does doubling the dilation each layer make the receptive field grow EXPONENTIALLY with depth, while a plain causal stack grows only linearly? (Measured empirically with a gradient probe.)_

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

# Reproduces WaveNet's core point (Sec 2.1) on a gradient probe: doubling dilation grows
# the receptive field exponentially with depth; plain (dilation-1) grows it linearly.
torch.manual_seed(0)

class CausalConv1d(nn.Module):
    def __init__(self, in_c, out_c, k=2, dilation=1):
        super().__init__()
        self.pad = (k - 1) * dilation
        self.conv = nn.Conv1d(in_c, out_c, k, dilation=dilation, padding=0)
    def forward(self, x):
        return self.conv(F.pad(x, (self.pad, 0)))      # LEFT pad only -> causal

def receptive_field(dilations, k=2, L=512):
    net = nn.Sequential(*[CausalConv1d(1, 1, k, d) for d in dilations])
    for p in net.parameters():
        nn.init.constant_(p, 1.0 if p.dim() > 1 else 0.0)
    inp = torch.zeros(1, 1, L, requires_grad=True)
    net(inp)[0, 0, -1].backward()                      # grad of LAST output sample
    return int((inp.grad[0, 0].abs() > 0).sum().item())

for depth in range(1, 9):
    dil  = [2 ** i for i in range(depth)]              # 1,2,4,... doubling
    flat = [1] * depth                                 # all dilation 1
    print(f"L={depth}  dilated R={receptive_field(dil)}  plain R={receptive_field(flat)}")
# L=1 2/2  L=2 4/3  L=3 8/4  L=4 16/5  ...  L=8 256/9   (dilated = 2^L, plain = 1+L)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation: dilated vs plain causal convolution. You have a 4-layer causal conv stack with
            kernel size $k=2$. Version A uses dilations $1,2,4,8$; version B uses dilation $1$ in every layer
            (plain causal). For each, what is the receptive field (how many past samples can influence the last
            output)? Which property of WaveNet does this isolate, and what would the gap be at 10 layers?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Apply $R = 1 + \sum_\ell (k-1)\,d_\ell$ with $k-1=1$ to version A:
                 $R_A = 1 + (1+2+4+8) = 16$. — _Dilations add their reaches; doubling them makes the partial sums a geometric series, so
                  $R_A = 2^{4} = 16$._
- Apply the same formula to version B (all $d=1$): $R_B = 1 + (1+1+1+1) = 5$. — _With no dilation each layer extends the reach by only $k-1=1$, so $R$ grows linearly with
                  depth: $R_B = 1 + L$._
- Compare: dilated reaches 16 past samples, plain reaches only 5, with identical depth
                 and parameter count. — _The only change is the dilation schedule, so the entire difference in memory is
                  attributable to dilation._
- Extrapolate to 10 layers: dilated $= 2^{10} = 1024$ (the paper's block size), plain $= 1+10 = 11$. — _Exponential vs linear: the gap explodes with depth, which is why WaveNet can reach thousands of
                  samples back with a shallow stack._

**Answer:** Version A (dilations $1,2,4,8$) has receptive field $1+(1+2+4+8)=\mathbf{16}=2^{4}$; version B
                 (all $d=1$) has receptive field $1+(1+1+1+1)=\mathbf{5}=1+L$. Same depth, same parameters &mdash;
                 the $16$-vs-$5$ gap is due entirely to dilation. At 10 layers the gap is $2^{10}=1024$ vs
                 $11$. This isolates dilated causal convolution as the mechanism behind WaveNet's long memory:
                 the receptive field grows exponentially with depth under doubling dilation but only
                 linearly without it (&sect;2.1). The CODEVIZ panel measures both empirically with a gradient
                 probe and confirms $16$ vs $5$.

</details>

**Problem 2.** Hand-compute one causal-conv step. A causal layer has $k=2$, $d=1$, weights $w=[w_0,w_1]=[1,\,3]$ (so
            $y_t = w_0 x_{t-1} + w_1 x_t$). For input $x=[2,\,5,\,1]$, left-pad by one zero and compute every
            output $y_t$. Verify no output uses a future sample.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Left-pad: the layer reaches back $(k-1)d=1$, so prepend one zero:
                 $x_{\text{pad}}=[0,\,2,\,5,\,1]$. — _Padding on the left (not the right) keeps the output length equal to the input and
                  guarantees causality._
- $y_1 = 1\cdot 0 + 3\cdot 2 = 6$. — _At $t=1$ there is no real past, so the left-pad zero fills $x_0$; only $x_1=2$ contributes._
- $y_2 = 1\cdot 2 + 3\cdot 5 = 2 + 15 = 17$. — _Sees $x_1=2$ (past) and $x_2=5$ (current) &mdash; no future term._
- $y_3 = 1\cdot 5 + 3\cdot 1 = 5 + 3 = 8$. — _Sees $x_2=5$ and $x_3=1$; again no $x_4$ exists or is used._

**Answer:** $y=[6,\,17,\,8]$. Each output $y_t = w_0 x_{t-1} + w_1 x_t$ uses only the current sample and one
                 earlier sample (the left-pad supplies the missing $x_0=0$), so no output ever reads a future
                 sample &mdash; the convolution is causal. This is the 1-D, time-only version of PixelCNN's
                 masked convolution (paper-pixelcnn).

</details>

**Problem 3.** The gated activation unit (Eq. 2) is
            $z=\tanh(W_f * x)\odot\sigma(W_g * x)$. At one feature position the content branch gives
            $\tanh(\cdot)=-0.6$ and the gate branch gives $\sigma(\cdot)=0.2$. What is $z$ there, and what would
            $z$ be if the gate were instead $\sigma(\cdot)=0.9$? What is the gate doing?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- First case: $z = (-0.6)\times 0.2 = -0.12$. — _The unit multiplies content by gate element-wise; a gate near $0$ nearly shuts the feature off._
- Second case: $z = (-0.6)\times 0.9 = -0.54$. — _A gate near $1$ lets almost the full content value pass (sign preserved)._
- Note the content value $-0.6$ was identical in both; only the gate changed the output. — _The sigmoid gate is a learned, input-dependent volume knob in $[0,1]$ on each content channel._

**Answer:** With gate $0.2$: $z=-0.12$ &mdash; the feature is mostly suppressed. With gate $0.9$:
                 $z=-0.54$ &mdash; nearly the full content passes (its sign kept). The sigmoid branch is a
                 per-element gate (a learned $[0,1]$ multiplier) controlling how much of the tanh content
                 reaches the output. The paper found this multiplicative gating "works significantly better ...
                 than the rectified linear activation function" for audio (&sect;2.3).

</details>